In [1]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def preprocess_img(path:str):
    org = cv.imread(path, cv.IMREAD_GRAYSCALE)
    hist_eq = cv.equalizeHist(org)
    return org, hist_eq

In [3]:
def kp_des(sift, hist_eq):
    kp, des = sift.detectAndCompute(hist_eq, None)
    return kp, des

In [4]:
def matching(flann, des1, des2):
    knn_matches = flann.knnMatch(des1, des2, 2)

    ratio_th = 0.6
    good_matches = []
    for m,n in knn_matches:
        if m.distance < ratio_th * n.distance:
            good_matches.append(m)
    return good_matches

In [5]:
def img_matching(sift, flann, hist1, hist2):
    kp1, des1 = kp_des(sift, hist1)
    kp2, des2 = kp_des(sift, hist2)
    good_matches = matching(flann, des1, des2)
    return kp1, kp2, good_matches

In [6]:
def draw_match(img1, img2, kp1, kp2, good_matches):
    img_matches = np.empty((max(img1.shape[0], img2.shape[0]), img1.shape[1]+img2.shape[1], 3), dtype=np.uint8)
    cv.drawMatches(img1, kp1, img2, kp2, good_matches, img_matches, flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    return img_matches

In [ ]:
org1, hist1 = preprocess_img('datasets/dataset_same_dog_and_cat/yo_Cats/Cat_Grumpy/Face/Cat_Grumpy_18_face.jpg')
org2, hist2 = preprocess_img('datasets/dataset_same_dog_and_cat/yo_Cats/Cat_Izzy/Face/Cat_Izzy_18_face.jpg')

sift = cv.SIFT_create()
flann = cv.DescriptorMatcher_create(cv.DescriptorMatcher_FLANNBASED)
kp1, kp2, good_matches = img_matching(sift, flann, hist1, hist2)
img_matches = draw_match(org1, org2, kp1, kp2, good_matches)
plt.imshow(img_matches)
plt.show()

In [12]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(sift, flann, dataset):
    # correct = 0
    pred = []
    y_true = []
    for hist1, hist2, label in dataset:
        y_true.append(label)
        _, _, good_matches = img_matching(sift, flann, hist1, hist2)
        if good_matches:
            pred.append(1)
        else:
            pred.append(0)
        # if good_matches and label == 1:
        #     correct += 1
        # elif not good_matches and label == 0:
        #     correct += 1
    else:
        # acc = correct/len(dataset)
        acc = accuracy_score(y_true, pred)
        f1 = f1_score(y_true, pred)
    return  acc, f1

In [8]:
from torch.utils.data import Dataset

class pet_dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        img1, img2 = row['img1'], row['img2']
        label = row['label']
        _, hist1 = preprocess_img(img1)
        _, hist2 = preprocess_img(img2)
        return hist1, hist2, label

In [9]:
import os
import random

def make_pairs(dataset_dir, num_pairs=1000):
    pairs = []

    dog_dir = os.path.join(dataset_dir, "yo_Dogs")
    dog_folders = [os.path.join(dog_dir, d) for d in os.listdir(dog_dir) 
                   if os.path.isdir(os.path.join(dog_dir, d))]
    
    cat_dir = os.path.join(dataset_dir, "yo_Cats")
    cat_folders = [os.path.join(cat_dir, d) for d in os.listdir(cat_dir) 
                   if os.path.isdir(os.path.join(cat_dir, d))]

    # เก็บไฟล์รูปของแต่ละ class (folder)
    class_to_images = {}
    for folder in dog_folders:
        face_folder = os.path.join(folder, "Face")
        images = [os.path.join(face_folder, f) for f in os.listdir(face_folder) 
                  if f.lower().endswith((".jpg", ".png", ".jpeg"))]
        class_to_images[face_folder] = images

    for folder in cat_folders:
        face_folder = os.path.join(folder, "Face")
        images = [os.path.join(face_folder, f) for f in os.listdir(face_folder)
                  if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        class_to_images[face_folder] = images

    all_classes = list(class_to_images.keys())

    for _ in range(num_pairs):
        # random decide positive หรือ negative
        if random.random() < 0.5:  
            # ---------- Positive pair ----------
            cls = random.choice(all_classes)
            imgs = random.sample(class_to_images[cls], 2)  # สองรูปจาก class เดียวกัน
            pairs.append((imgs[0], imgs[1], 1))
        else:
            # ---------- Negative pair ----------
            cls1, cls2 = random.sample(all_classes, 2)
            img1 = random.choice(class_to_images[cls1])
            img2 = random.choice(class_to_images[cls2])
            pairs.append((img1, img2, 0))

    return pairs

# ===============================
# Example run
# ===============================
dataset_dir = "datasets\\dataset_same_dog_and_cat"
pairs = make_pairs(dataset_dir, num_pairs=5000)

pairs

[('datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Coco\\Face\\Dog_Coco_2_face.jpg',
  'datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Coco\\Face\\Dog_Coco_11_face.jpg',
  1),
 ('datasets\\dataset_same_dog_and_cat\\yo_Cats\\Cat_Venus\\Face\\Cat_Venus_1_face.jpg',
  'datasets\\dataset_same_dog_and_cat\\yo_Cats\\Cat_Strellar\\Face\\Cat_Strellar_2_face.jpg',
  0),
 ('datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Coco\\Face\\Dog_Coco_12_face.jpg',
  'datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Coco\\Face\\Dog_Coco_19_face.jpg',
  1),
 ('datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Parker\\Face\\Dog_Parker_19_face.jpg',
  'datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Milkykun\\Face\\Dog_Milkykun_14_face.jpg',
  0),
 ('datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Deko\\Face\\Dog_Deko_10_face.jpg',
  'datasets\\dataset_same_dog_and_cat\\yo_Dogs\\Dog_Deko\\Face\\Dog_Deko_4_face.jpg',
  1),
 ('datasets\\dataset_same_dog_and_cat\\yo_Cats\\Cat_Jemua\\Face\\Cat_Jemua_20_face

In [13]:
import pandas as pd

df = pd.DataFrame(pairs, columns=['img1', 'img2', 'label'])
dataset = pet_dataset(df)
sift = cv.SIFT_create()
flann = cv.DescriptorMatcher_create(cv.DescriptorMatcher_FLANNBASED)
evaluate(sift, flann, dataset)

(0.643, 0.5213193885760258)